# Influence and hub test on Kaggle or Colab

Runs `models/explainability/run_gnn.sh` on a GPU for the models in `MODELS`:

- **DCRNN and Graph WaveNet** (fixed adjacency): trains each on a degree-preserving random graph, then runs `influence.py` and `hubs.py` on the real-graph and random-graph models.
- **Adaptive-GVAR**: trains the random-graph model with `models/GVAR/train_adaptive.py` (30 epochs). Then it runs the analysis twice: once removing connectome edges only, as for the other models, and once (`--learned`) also removing each neuron from the graphs the model learns.

See `models/explainability/README.md` for what is measured.

Before running:

1. Push the `explainability-analysis` branch. The trained connectome models (`models/stgnn/runs/*/best.pt`, `models/GVAR/checkpoints/adaptive_gvar_sparsemax/best.pt`) come with the clone.
2. Upload `processed/session_4_7_cave_proofread` (156 MB, not in git). On Kaggle, add it as a dataset. On Colab, put the folder on Google Drive.
3. Turn on a GPU. On Kaggle pick **T4** (recent PyTorch builds do not support the P100) and turn on internet access.
4. The repository is private, so store a GitHub token as the secret `GITHUB_TOKEN` (Kaggle: *Add-ons → Secrets*, then tick it for this notebook; Colab: the key icon, then allow notebook access). Use a classic token with the `repo` scope: fine-grained tokens cannot reach a personal repository you are only a collaborator on.

Each finished step is copied to `SAVE_TO`. If the session ends, run the notebook again and finished steps are skipped. On Kaggle, use *Save Version → Save & Run All* so the run continues in the background and `/kaggle/working` is kept. A new version starts with an empty `/kaggle/working`: to resume, add the output of the earlier version as an input and copy its `results` folder to `SAVE_TO` first.

In [ ]:
import glob
import os

KAGGLE = os.path.exists('/kaggle')
REPO = 'https://github.com/G00DCOD3R/Neural-Forecasting.git'
BRANCH = 'explainability-analysis'
STRIDE = 4  # every 4th test window, as in the README
MODELS = 'graph_wavenet adaptive_gvar'  # DCRNN is done; add dcrnn to rerun it

if KAGGLE:
    # wherever Kaggle mounted the uploaded session (the folder holding X_features.npy, conditions.csv, edge_index.pt, ...)
    found = glob.glob('/kaggle/input/**/X_features.npy', recursive=True)
    assert len(found) == 1, f'expected one X_features.npy under /kaggle/input, found {found}'
    DATA = os.path.dirname(found[0])
    SAVE_TO = '/kaggle/working/results'
    WORK = '/tmp/Neural-Forecasting'
else:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA = '/content/drive/MyDrive/neural-forecasting/session_4_7_cave_proofread'
    SAVE_TO = '/content/drive/MyDrive/neural-forecasting/results'
    WORK = '/content/Neural-Forecasting'

assert os.path.exists(f'{DATA}/X_features.npy'), f'processed session not found in {DATA}'
print('data:', DATA)

In [ ]:
# The repository is private: clone with a GitHub token stored as the secret GITHUB_TOKEN
# (Kaggle: Add-ons -> Secrets; Colab: the key icon in the left bar)
if not os.path.exists(WORK):
    if KAGGLE:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    else:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
    url = REPO.replace('https://', f'https://{token}@')
    !GIT_TERMINAL_PROMPT=0 git clone --depth 1 -b {BRANCH} {url} {WORK}
    !git -C {WORK} remote set-url origin {REPO}  # do not leave the token in .git/config
os.makedirs(f'{WORK}/processed', exist_ok=True)
if not os.path.exists(f'{WORK}/processed/session_4_7_cave_proofread'):
    os.symlink(DATA, f'{WORK}/processed/session_4_7_cave_proofread')

Same versions as `models/stgnn/requirements.txt` (torch < 2.13), with CUDA 12.6.

In [ ]:
!pip install -q torch==2.12.1 --index-url https://download.pytorch.org/whl/cu126
!pip install -q torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.12.1+cu126.html
!pip install -q torch-geometric torch-spatiotemporal
!python -c "import torch, tsl; print(torch.__version__, tsl.__version__, 'cuda', torch.cuda.is_available())"

Bring back steps finished in an earlier session, then run everything that is missing.

In [ ]:
if os.path.exists(SAVE_TO):
    !cp -rn {SAVE_TO}/. {WORK}/models/
!cd {WORK} && SAVE_TO={SAVE_TO} STRIDE={STRIDE} bash models/explainability/run_gnn.sh {MODELS}

In [ ]:
import json
from IPython.display import Image, display

runs = ['dcrnn_cave_proofread_dcrnn_h1', 'graph_wavenet_cave_proofread_gwn_fixed_h1',
        'adaptive_gvar_sparsemax', 'adaptive_gvar_sparsemax_learned']
for run in runs:
    out = f'{WORK}/models/explainability/runs/{run}'
    if not os.path.exists(f'{out}/hubs.json'):
        continue
    h = json.load(open(f'{out}/hubs.json'))
    c = h['concentration']
    print(f"{run}: top-20% share {c['influence_drop'][0]:.1%} [{c['influence_drop'][1]:.1%}, {c['influence_drop'][2]:.1%}], "
          f"random graph {c['null_influence_drop'][0]:.1%}, out-degree {c['wiring_out_degree']:.1%}, "
          f"evidence for hubs: {h['evidence_for_hubs']}")
    display(Image(f'{out}/concentration.png', width=400))

Download `SAVE_TO` and copy its contents into `models/` in your local repository. It holds `stgnn/runs/*_rewired_h1`, `GVAR/checkpoints/adaptive_gvar_sparsemax_rewired` and `explainability/runs/*`.